# Anime Subtitle Generator (WhisperJAV, T4) — enhanced

**Pipeline:** `qwen` + `anime-whisper` + WhisperSeg, sensitivity `aggressive`.

**Enhancements over v1:** live heartbeat during long ASR phases, VRAM guard before Phase 5,
explicit model-refresh interval, output MILEAGE verification, checkpointed install/download,
and failure-aware auto-disconnect (keeps the session alive if something breaks so you can read the log).

Run Step 1 -> 2 -> 3 -> 4. Step 5 only if you chose English.

In [ ]:
#@title Step 1: Settings { display-mode: "form" }

#@markdown ### Source
SOURCE_URL  = "http://violetstreamio.com/MRVIOLETSTREAMBOT-AgAD7R15412" #@param {type:"string"}
TARGET_NAME = "Welsh.and.Shedar.S01E01.1080p.AMZN.WEB-DL.JPN.DDP2.0.H.264-ToonsHub.mkv" #@param {type:"string"}
EXPECTED_MB = 742 #@param {type:"integer"}

#@markdown ### Subtitle output
subtitle_language = "Japanese" #@param ["Japanese", "English (AI translate)"]

#@markdown ### Anime pipeline
sensitivity       = "aggressive" #@param ["conservative", "balanced", "aggressive"]
qwen_generator    = "anime-whisper" #@param ["anime-whisper", "qwen3"]
speech_segmenter  = "whisperseg" #@param ["whisperseg", "ten", "firered-vad", "silero-v3.1", "silero-v4.0", "silero-v6.2"]

#@markdown ### Advanced
#@markdown `model_refresh_minutes` — WhisperJAV reloads the recognizer every N minutes of audio
#@markdown to bound a known fault where it stops returning text. Default is 20; a single episode
#@markdown is under 25 so setting it higher avoids a mid-file reload. Set to 0 to disable.
model_refresh_minutes = 25 #@param {type:"integer"}

#@markdown `heartbeat_seconds` — how often to print elapsed time and VRAM during long phases.
heartbeat_seconds = 30 #@param {type:"integer"}

#@markdown ### Translation (only used when output is "English (AI translate)")
translation_provider = "local" #@param ["local", "gemini", "deepseek", "openrouter", "claude", "gpt", "groq"]
translation_model    = "gemma-9b" #@param {type:"string"}
translation_api_key  = "" #@param {type:"string"}

#@markdown ### Credit line
closing_credit = "Subs by WhisperJAV" #@param {type:"string"}

WHISPERJAV_RELEASE = "v1.9.3"
language_code = "native" if subtitle_language == "Japanese" else "llm"

print("Ready.")
print(f"  Source:      {SOURCE_URL}")
print(f"  Filename:    {TARGET_NAME}")
print(f"  Pipeline:    qwen / {qwen_generator} / {speech_segmenter} / {sensitivity}")
print(f"  Refresh:     every {model_refresh_minutes or 'never'} min of audio")
print(f"  Heartbeat:   every {heartbeat_seconds}s")
print(f"  Output:      {subtitle_language}")

In [ ]:
#@title Step 2: Download the episode from your server { display-mode: "form" }
# ENHANCED: checkpointed — re-running this cell is safe; it skips an already-complete download.

import os, subprocess, shlex, time, hashlib
from pathlib import Path
from IPython.display import display, HTML

def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")

WORK_DIR = Path("/content/media")
WORK_DIR.mkdir(parents=True, exist_ok=True)
dest_file = WORK_DIR / TARGET_NAME

# ENHANCED: 2% tolerance instead of 50 MB, plus a .done marker so we don't rely on size alone.
done_marker = WORK_DIR / (TARGET_NAME + ".done")
size_ok = dest_file.exists() and dest_file.stat().st_size >= EXPECTED_MB * 1e6 * 0.98
if size_ok and done_marker.exists():
    status(f"Already downloaded: {dest_file.name} ({dest_file.stat().st_size/1e6:.0f} MB)")
    WHISPERJAV_MEDIA_FILE = dest_file
    raise SystemExit()

print("Probing server...")
probe = subprocess.run(["curl", "-sI", "--max-time", "20", SOURCE_URL],
                       capture_output=True, text=True)
headers = probe.stdout.lower()
supports_ranges = "accept-ranges: bytes" in headers
content_len = None
for line in headers.splitlines():
    if line.startswith("content-length:"):
        try:
            content_len = int(line.split(":", 1)[1].strip())
        except ValueError:
            pass

if content_len:
    print(f"  Server reports: {content_len/1e6:.1f} MB")
print(f"  Range requests supported: {supports_ranges}")

if supports_ranges:
    if subprocess.run(["which", "aria2c"], capture_output=True).returncode != 0:
        print("Installing aria2...")
        subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
    print("Downloading with aria2c (16 connections)...\n")
    cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M",
           "--file-allocation=none", "--console-log-level=warn",
           "--summary-interval=5", "--auto-file-renaming=false",
           "--allow-overwrite=true", "--continue=true",
           "-d", str(WORK_DIR), "-o", TARGET_NAME, SOURCE_URL]
else:
    print("Server does not support range requests — single-stream download.\n")
    cmd = ["wget", "--continue", "--tries=10", "--waitretry=5",
           "--timeout=60", "--progress=dot:giga", "-O", str(dest_file), SOURCE_URL]

print("Command:", shlex.join(cmd), "\n")
start = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
elapsed = time.time() - start

if proc.returncode != 0:
    status(f"Download failed (exit {proc.returncode})", False)
    raise SystemExit()

if not dest_file.exists():
    alt = WORK_DIR / (TARGET_NAME + ".1")
    if alt.exists():
        alt.rename(dest_file)
    else:
        raise SystemExit(f"File not found: {dest_file}")

size_mb = dest_file.stat().st_size / 1e6
mbps = size_mb / elapsed if elapsed > 0 else 0
status(f"Downloaded {dest_file.name} ({size_mb:.0f} MB in {elapsed:.0f}s = {mbps:.1f} MB/s)")

# ENHANCED: verify size within 2%, and write a .done marker.
if size_mb < EXPECTED_MB * 0.98:
    display(HTML(f'<div style="padding:8px;background:#fef9c3;border-radius:4px;'
                 f'border-left:2px solid #ca8a04;font-size:11px"><b>Warning:</b> '
                 f'{size_mb:.0f} MB, expected ~{EXPECTED_MB} MB. Re-run this cell to resume.</div>'))
else:
    done_marker.write_text("ok")

WHISPERJAV_MEDIA_FILE = dest_file
print("\nRun Step 3 next.")

In [ ]:
#@title Step 3: Install WhisperJAV and transcribe (with heartbeat) { display-mode: "form" }
# ENHANCED: checkpointed install, pre-ASR VRAM guard, live heartbeat during transcription,
# explicit --model-refresh-audio-minutes, exit-code surfacing.

import os, sys, subprocess, shlex, time, threading
from pathlib import Path
from IPython.display import display, HTML

os.environ['MPLBACKEND'] = 'Agg'
os.environ['TORCH_HUB_TRUST_REPO'] = '1'

def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")

def section(title):
    print(f"\n{'-'*46}\n{title}\n{'-'*46}")

if 'WHISPERJAV_MEDIA_FILE' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 2 first.</b></div>'))
    raise SystemExit()

section("GPU CHECK")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    status("No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.", False)
    raise SystemExit()
status(f"GPU: {gpu.stdout.strip()}")

section(f"INSTALLING WHISPERJAV {WHISPERJAV_RELEASE}")
install_start = time.time()
REPO_URL = "https://github.com/meizhong986/WhisperJAV.git"
REPO_PATH = "/content/WhisperJAV"
SCRIPT_PATH = f"{REPO_PATH}/installer/install_colab.sh"

# ENHANCED: also check the CLI is on PATH, not just the import.
import_check = subprocess.run(["python3", "-c", "import whisperjav"], capture_output=True)
cli_check = subprocess.run(["which", "whisperjav"], capture_output=True)
if import_check.returncode == 0 and cli_check.returncode == 0:
    status("WhisperJAV already installed (skipping)")
else:
    if not os.path.exists(REPO_PATH):
        print(f"Cloning {WHISPERJAV_RELEASE}...")
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", WHISPERJAV_RELEASE,
                            REPO_URL, REPO_PATH], capture_output=True, text=True)
        if r.returncode != 0:
            status(f"Clone failed: {r.stderr.strip()}", False)
            raise SystemExit()
    if not os.path.exists(SCRIPT_PATH):
        status(f"Install script not found: {SCRIPT_PATH}", False)
        raise SystemExit()

    env = {**os.environ,
           "PATH": f"{os.environ.get('PATH', '')}:{os.path.expanduser('~/.local/bin')}",
           "WHISPERJAV_REF": WHISPERJAV_RELEASE}
    p = subprocess.Popen(["bash", SCRIPT_PATH], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(p.stdout.readline, ''):
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        status("Installation failed", False)
        raise SystemExit()
status(f"Install finished ({time.time()-install_start:.0f}s)")

# ENHANCED: pre-ASR VRAM guard. Fail early with a clear message instead of an OOM deep in Phase 5.
section("VRAM GUARD")
try:
    import torch, gc
    gc.collect()
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    free_gb = free_b / 1e9
    total_gb = total_b / 1e9
    print(f"  Free VRAM: {free_gb:.1f} GB / {total_gb:.1f} GB")
    if free_gb < 8:
        status(f"Only {free_gb:.1f} GB free. anime-whisper + WhisperSeg want ~8 GB. "
               f"Restart the runtime (Runtime -> Restart) and re-run from Step 2.", False)
        raise SystemExit()
    status("Enough VRAM for the anime-whisper pipeline")
except ImportError:
    print("  torch not importable yet — skipping the VRAM guard")

section("TRANSCRIBING")
out_dir = WHISPERJAV_MEDIA_FILE.parent
cmd = ["whisperjav", str(WHISPERJAV_MEDIA_FILE),
       "--output-dir", str(out_dir),
       "--mode", "qwen",
       "--qwen-sensitivity", sensitivity,
       "--qwen-generator", qwen_generator,
       "--qwen-segmenter", speech_segmenter,
       "--subs-language", language_code]

# ENHANCED: explicit refresh interval so a 24-minute file doesn't reload the model mid-run.
if model_refresh_minutes and model_refresh_minutes > 0:
    cmd += ["--model-refresh-audio-minutes", str(model_refresh_minutes)]
else:
    cmd += ["--model-refresh-audio-minutes", "0"]

print("Command:", shlex.join(cmd), "\n")

# ENHANCED: heartbeat thread prints elapsed + GPU state while the subprocess runs.
start = time.time()
stop_heartbeat = threading.Event()

def heartbeat():
    while not stop_heartbeat.is_set():
        stop_heartbeat.wait(heartbeat_seconds)
        if stop_heartbeat.is_set():
            break
        elapsed = time.time() - start
        mins, secs = divmod(int(elapsed), 60)
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                                  "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=5)
            vram, util = out.stdout.strip().split(", ") if out.stdout.strip() else ("?", "?")
            print(f"\n[heartbeat {mins:02d}:{secs:02d}]  VRAM {vram} MiB  GPU {util}%", flush=True)
        except Exception:
            print(f"\n[heartbeat {mins:02d}:{secs:02d}]", flush=True)

hb = threading.Thread(target=heartbeat, daemon=True)
hb.start()

p = subprocess.Popen(shlex.join(cmd), shell=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
last_lines = []
for line in p.stdout:
    print(line, end='', flush=True)
    last_lines.append(line.rstrip())
    if len(last_lines) > 200:
        last_lines.pop(0)
p.wait()
stop_heartbeat.set()
hb.join(timeout=2)

if p.returncode != 0:
    status(f"Transcription failed (exit {p.returncode})", False)
    print("\nLast 20 log lines:")
    for ln in last_lines[-20:]:
        print("  " + ln)
    raise SystemExit()

srts = sorted([f for f in out_dir.glob('*.srt')
               if '.pass1.' not in f.name and '.pass2.' not in f.name])
status(f"Created {len(srts)} subtitle file(s)")
for s in srts:
    print(f"  - {s.name} ({s.stat().st_size} bytes)")

WHISPERJAV_SRTS = srts
WHISPERJAV_OUT_DIR = out_dir

In [ ]:
#@title Step 3b: Verify subtitle output (MILEAGE) { display-mode: "form" }
# ENHANCED: reads the last cue timestamp and compares to the audio length.
# Catches the "subtitles stop at 5% of the file" case before you notice it in the player.

import re, subprocess
from IPython.display import display, HTML

if 'WHISPERJAV_SRTS' not in dir() or not WHISPERJAV_SRTS:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>No subtitles to verify — run Step 3 first.</b></div>'))
    raise SystemExit()

ffprobe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(WHISPERJAV_MEDIA_FILE)
], capture_output=True, text=True)
audio_len = float(ffprobe.stdout.strip()) if ffprobe.stdout.strip() else 0

time_re = re.compile(r"(\d+):(\d+):(\d+),(\d+)")

for srt in WHISPERJAV_SRTS:
    text = srt.read_text(encoding='utf-8', errors='replace')
    cues = len(re.findall(r"^\d+\s*$", text, flags=re.MULTILINE))
    matches = time_re.findall(text)
    if matches:
        h, m, s, ms = map(int, matches[-1])
        last_sec = h*3600 + m*60 + s + ms/1000
    else:
        last_sec = 0
    mileage = (last_sec / audio_len * 100) if audio_len else 0
    verdict = "done" if mileage >= 75 else ("suspect" if mileage >= 25 else "empty")
    color = {"done": "#16a34a", "suspect": "#ca8a04", "empty": "#991b1b"}[verdict]
    display(HTML(f'<div style="padding:8px 10px;background:#f9fafb;border-radius:4px;'
                 f'border-left:2px solid {color};font-size:11px">'
                 f'<b>{verdict}</b> &nbsp; {srt.name} &nbsp; '
                 f'{cues} cues, last at {last_sec:.0f}s of {audio_len:.0f}s ({mileage:.0f}%)</div>'))

In [ ]:
#@title Step 4: Add credit + save to Drive { display-mode: "form" }
# ENHANCED: only disconnects on success. On failure, the session stays alive so you can read the log.

import shutil, time, sys
from pathlib import Path
from google.colab import drive
from IPython.display import display, HTML

def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")

if 'WHISPERJAV_SRTS' not in dir() or not WHISPERJAV_SRTS:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>No subtitles to save — run Step 3 first.</b></div>'))
    raise SystemExit()

# Add credit line (same as before).
for srt in WHISPERJAV_SRTS:
    if closing_credit:
        try:
            content = srt.read_text(encoding='utf-8')
            content += f"\n9999\n23:59:58,000 --> 23:59:59,000\n{closing_credit}\n"
            srt.write_text(content, encoding='utf-8')
        except Exception as e:
            print(f"  credit failed on {srt.name}: {e}")
status(f"Credit added to {len(WHISPERJAV_SRTS)} file(s)")

drive.mount('/content/drive', force_remount=False)
drive_dir = Path("/content/drive/MyDrive/WhisperJAV")
drive_dir.mkdir(parents=True, exist_ok=True)

for f in list(WHISPERJAV_SRTS) + [WHISPERJAV_MEDIA_FILE]:
    target = drive_dir / f.name
    print(f"Copying {f.name} -> Drive...")
    shutil.copy2(f, target)

status(f"Saved {len(WHISPERJAV_SRTS)} .srt + media to Google Drive/WhisperJAV/")
display(HTML(f'<div style="padding:10px;background:#f0fdf4;border-radius:4px;'
             f'border-left:2px solid #16a34a;font-size:12px"><b>Done.</b> '
             f'Files are in your Drive under <code>WhisperJAV/</code>.</div>'))

print("\nDisconnecting in 10 seconds to save GPU credits...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass

In [ ]:
#@title Step 5: AI Translation (only if output is "English (AI translate)") { display-mode: "form" }
# ENHANCED: same as v1, but reports per-file status and copies results back to Drive.

import os, shlex, subprocess, time, shutil
from pathlib import Path
from IPython.display import display, HTML

if 'WHISPERJAV_SRTS' not in dir() or not WHISPERJAV_SRTS:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 3 first.</b></div>'))
    raise SystemExit()

if subtitle_language == "Japanese":
    display(HTML('<div style="padding:8px;background:#f0f9ff;border-radius:4px;'
                 'border-left:2px solid #3b82f6;font-size:11px"><b>Skipped:</b> '
                 'output is Japanese only.</div>'))
    raise SystemExit()

if translation_provider != "local" and not translation_api_key:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>No API key.</b> Set one in Step 1, or use local.</div>'))
    raise SystemExit()

if translation_provider != "local":
    env_map = {"gemini": "GEMINI_API_KEY", "deepseek": "DEEPSEEK_API_KEY",
               "openrouter": "OPENROUTER_API_KEY", "claude": "ANTHROPIC_API_KEY",
               "gpt": "OPENAI_API_KEY", "groq": "GROQ_API_KEY"}
    os.environ[env_map.get(translation_provider, "API_KEY")] = translation_api_key

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
drive_dir = Path("/content/drive/MyDrive/WhisperJAV")

print(f"Provider: {translation_provider}")
print(f"Files: {len(WHISPERJAV_SRTS)}\n")

succeeded = []
failed = []
for srt in WHISPERJAV_SRTS:
    print(f"Translating {srt.name}...")
    cmd = ["whisperjav-translate", "-i", str(srt),
           "--provider", translation_provider,
           "-t", "english", "--tone", "standard", "--stream"]
    if translation_model:
        cmd += ["--model", translation_model]
    p = subprocess.Popen(shlex.join(cmd), shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(f"  {line}", end='', flush=True)
    p.wait()
    if p.returncode == 0:
        succeeded.append(srt)
    else:
        failed.append(srt)
        print(f"  FAILED: {srt.name}")

for f in WHISPERJAV_OUT_DIR.glob('*.en.srt'):
    shutil.copy2(f, drive_dir / f.name)

if failed:
    display(HTML(f'<div style="padding:10px;background:#fef9c3;border-radius:4px;'
                 f'border-left:2px solid #ca8a04;font-size:12px"><b>Partly done:</b> '
                 f'{len(succeeded)} translated, {len(failed)} failed. '
                 f'Japanese .srt files are still in Drive/WhisperJAV/.</div>'))
else:
    display(HTML('<div style="padding:10px;background:#f0fdf4;border-radius:4px;'
                 'border-left:2px solid #16a34a;font-size:12px"><b>Translation done.</b> '
                 'English .srt files are in Google Drive/WhisperJAV/.</div>'))

print("\nDisconnecting in 10 seconds...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass